# einops-rearrange-flatten composite — cx26: flatten patch group then reduce within the flattened axis

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-rearrange-flatten`, `einops-reduce`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "einops-rearrange-flatten"
DD_ATOM_IDS = ["einops-rearrange-flatten", "einops-reduce"]
DD_SUBTOPICS = ["Einops: Rearrange-as-flatten", "Einops: Reduce"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Vision pipelines often `rearrange` a CNN feature map to GROUP spatial sub-blocks (e.g. patches, pooling windows) into a new flattened axis — then `reduce` along that flattened axis to get a per-block summary statistic. The two atoms naturally chain: rearrange composes the axes you want to summarize, reduce collapses them.

Worked example: 2x2 average pooling. Rearrange `'b c (h h2) (w w2) -> b c h w (h2 w2)'` to group the 2x2 spatial blocks into a final flattened axis of size 4, then `reduce(..., 'b c h w n -> b c h w', 'mean')` to average within each block. The composition makes pooling expressible as a shape transform + a reduction — no `F.avg_pool2d` needed.

### Composite Exercise — flatten patch group then reduce within the flattened axis

**Atoms exercised together**: `einops-rearrange-flatten`, `einops-reduce`

Implement `cx26_block_pool_via_flatten_reduce(x, block, reduction)` that performs `block` x `block` non-overlapping pooling over the spatial axes of an `(B, C, H, W)` tensor.

1. **Rearrange** the input with `'b c (h h2) (w w2) -> b c h w (h2 w2)'` (substitute `h2=w2=block`). This groups each `block x block` spatial block into a final flattened axis of length `block * block`.
2. **Reduce** along that flattened axis with `reduce(..., 'b c h w n -> b c h w', reduction)`. `reduction` is one of `'mean'`, `'max'`, `'sum'`.

Return the pooled `(B, C, H // block, W // block)` tensor.

Cross-check against `F.avg_pool2d` / `F.max_pool2d` so the composition is verified, not just typed.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx26_block_pool_via_flatten_reduce(x, block, reduction):
    raise NotImplementedError

def _test_cx26():
    import torch.nn.functional as F

    # Case A: 2x2 average pool — cross-check against F.avg_pool2d.
    x = t.randn(2, 3, 8, 8)
    out = cx26_block_pool_via_flatten_reduce(x, block=2, reduction='mean')
    assert tuple(out.shape) == (2, 3, 4, 4), f'got {tuple(out.shape)}'
    ref = F.avg_pool2d(x, kernel_size=2)
    assert t.allclose(out, ref, atol=1e-6), f'avg-pool mismatch (max diff {(out-ref).abs().max()})'

    # Case B: 2x2 max pool.
    out_max = cx26_block_pool_via_flatten_reduce(x, block=2, reduction='max')
    ref_max = F.max_pool2d(x, kernel_size=2)
    assert t.allclose(out_max, ref_max), 'max-pool mismatch'

    # Case C: 4x4 sum reduction over a single batch/channel — hand-check exact sums.
    y = t.arange(16.0).reshape(1, 1, 4, 4)
    out_sum = cx26_block_pool_via_flatten_reduce(y, block=4, reduction='sum')
    assert tuple(out_sum.shape) == (1, 1, 1, 1)
    assert out_sum.item() == sum(range(16)), f'got {out_sum.item()}'

    # Case D: non-square spatial — block=2 on (B,C,6,10).
    z = t.randn(1, 2, 6, 10)
    out_z = cx26_block_pool_via_flatten_reduce(z, block=2, reduction='mean')
    assert tuple(out_z.shape) == (1, 2, 3, 5)
    assert t.allclose(out_z, F.avg_pool2d(z, kernel_size=2), atol=1e-6)
    _dd_passed.add('cx26')

_test_cx26()

<details><summary>Show solution — cx26</summary>

```python
def cx26_block_pool_via_flatten_reduce(x, block, reduction):
    # Atom A (rearrange-flatten): group each block x block window into a final flat axis.
    grouped = rearrange(
        x, 'b c (h h2) (w w2) -> b c h w (h2 w2)', h2=block, w2=block
    )
    # Atom B (reduce): collapse the flattened block axis with the requested reduction.
    return reduce(grouped, 'b c h w n -> b c h w', reduction)
```

Rearrange-flatten followed by reduce is the workhorse pattern for spatial pooling in einops code. Note how the named axis `n` in the second pattern is just a placeholder for the grouped `(h2 w2)` axis — einops doesn't care what you call it, only that exactly one axis is collapsed. Swapping `reduction` between `'mean'`, `'max'`, `'sum'` recovers the three standard pool ops without a separate API per kind.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx26'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx26',
        'subtopics': ["Einops: Rearrange-as-flatten", "Einops: Reduce"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()